# Transaction Investigation & Dispute — EDA Addendum

Pregunta: ¿hay suficiente señal, cobertura y calidad para justificar discovery?

Este notebook complementa, sin reemplazar, 02_eda.ipynb y reports/eda/EDA_FINDINGS.md.
No construye journeys, matching, modelos, thresholds, agentes ni una arquitectura.
Los resultados completos y denominadores se exportan a artifacts/eda_transaction_dispute/.
Las tablas pequeñas usan pandas solo para presentación; los cálculos completos usan DuckDB.
No se consultan fuentes externas, no se imputan datos ni se eliminan extremos.


In [1]:
from pathlib import Path
import sys
from contextlib import ExitStack
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = ROOT.parent
assert (ROOT / "AGENTS.md").exists()
assert sys.version_info >= (3, 12)
sys.path.insert(0, str(ROOT))
from src.dispute_eda import (ART, sources, fraud_label, fraud_scores, taxonomy,
    claimed_amount, transaction_evidence, geography, status_evidence, outcomes, label_distributions)
from src.dispute_reporting import build_report, validate_artifacts

def show(name, limit=30):
    # Strings preserve codes such as 00 and explicit NULL display.
    frame = pd.read_csv(ART / name, dtype=str, keep_default_na=False)
    display(frame.head(limit))
    if len(frame) > limit:
        print(f"Shown {limit}/{len(frame)} aggregate rows; full table: {name}")

resources = ExitStack()
con, context = resources.enter_context(sources())
display(context)
show("population_cutoff_audit.csv")


Verificando SHA-256 de 3291 fuentes de los tres dominios utilizados


Recuperando únicamente complaint_id/resolution: columna excluida de la caché del EDA general


{'execution_started': '2026-10-01T23:44:03.623637-05:00',
 'feature_cutoff_inclusive': '2026-06-17',
 'primary_population': 'Full observed extract; row dates beyond the inherited cutoff are counted separately.',
 'availability_warning': 'Labels and snapshot outcomes have no known availability timestamp; a row before cutoff does not prove its label was available.',
 'source_manifest_sha256': 'edaa6b4db2c0053ead935ffc30d0d2928d0d4769bfa8e7b9d880e57043827074',
 'source_files_sha256_checked': 3291,
 'reference_files_sha256': {'notebooks/02_eda.ipynb': 'b0929f98f5da16e6488b03c2edf8a3de16d74d35d9acd55f63a6d22c2af3fabf',
  'reports/eda/EDA_FINDINGS.md': 'f043c56fd8d65d81a0b6e10ed68b4b90ff184d5bf6a53f429353cde2e0955a62',
  'reports/eda/analysis_context.json': 'de3707365afe981743c6c1c479aa073133e3afbdf1928ce4ebc00308ea504e4d',
  'reports/eda/semantic_validation.csv': 'fde21ed18f9df35c273d4c870115a7e3f8d86195bea15dcfc37092d56e71faa3',
  'reports/eda/transcript_coverage.csv': 'dc04bd964bbb130e135

,dataset,records,after_cutoff
0,transactions,4425008,1352
1,complaints,67095,29
2,call_center_interactions,686296,175


## 1. Transaction fraud label

Población principal: extracto completo. fraud_rate = True / labels booleanas válidas.
Se exporta también el subconjunto cuya fecha de transacción no supera el cutoff heredado;
esto no prueba disponibilidad histórica de la etiqueta. Las siete dimensiones retienen NULLs.


In [2]:
fraud_label(con, context)
show("fraud_label_summary.csv")
for dimension in ("transaction_status", "transaction_type", "transaction_category",
                  "channel", "transaction_country", "currency", "merchant_category"):
    show(f"fraud_by_{dimension}.csv")


1. Label de fraude y segmentaciones completadas


,population,total_transactions,valid_is_fraud,null_is_fraud,invalid_is_fraud,fraud_transactions,nonfraud_transactions,fraud_rate
0,full_extract,4425008,4425008,0,0,4316,4420692,0.0009753654682658201
1,row_date_at_cutoff,4423656,4423656,0,0,4315,4419341,0.0009754375114158967


,transaction_status,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,Approved,4070681,4070681,0,0,3986,4066695,0.0009791973382340695
1,Declined,221234,221234,0,0,215,221019,0.0009718216910601444
2,Pending,88343,88343,0,0,79,88264,0.0008942417622222474
3,Reversed,44750,44750,0,0,36,44714,0.0008044692737430168


,transaction_type,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,Purchase,1083406,1083406,0,0,1086,1082320,0.001002394300936122
1,Withdrawal,964673,964673,0,0,951,963722,0.0009858262851764276
2,Transfer,896438,896438,0,0,798,895640,0.0008901898402343498
3,Payment,738964,738964,0,0,730,738234,0.000987869503791795
4,Deposit,609409,609409,0,0,607,608802,0.0009960469897884671
5,Adjustment,132118,132118,0,0,144,131974,0.0010899347552945095


,transaction_category,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,,2693520,2693520,0,0,2603,2690917,0.0009663934182779411
1,Food,432468,432468,0,0,424,432044,0.0009804193605075983
2,Services,345370,345370,0,0,341,345029,0.000987346903321076
3,Other,260518,260518,0,0,267,260251,0.0010248811982281455
4,Transport,260316,260316,0,0,243,260073,0.0009334808463559674
5,Entertainment,259643,259643,0,0,253,259390,0.0009744148696479397
6,Health,173173,173173,0,0,185,172988,0.0010682958659837274


,channel,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,POS,1548161,1548161,0,0,1467,1546694,0.0009475758658175732
1,ATM,1328334,1328334,0,0,1331,1327003,0.001002007025341518
2,Web,663445,663445,0,0,656,662789,0.0009887782709945812
3,App,663414,663414,0,0,634,662780,0.0009556626782069718
4,Branch,132495,132495,0,0,143,132352,0.0010792860107928601
5,Transfer,89159,89159,0,0,85,89074,0.0009533529985755784


,transaction_country,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,México,2105794,2105794,0,0,2082,2103712,0.0009887006991187173
1,Colombia,1289503,1289503,0,0,1198,1288305,0.0009290401030474532
2,Argentina,867561,867561,0,0,864,866697,0.0009958953894884624
3,USA,40621,40621,0,0,38,40583,0.0009354767238620418
4,Spain,40542,40542,0,0,39,40503,0.0009619653692467072
5,Mexico,40515,40515,0,0,50,40465,0.001234110823151919
6,Brazil,40472,40472,0,0,45,40427,0.0011118798181458786


,currency,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,USD,2437979,2437979,0,0,2414,2435565,0.000990164394361067
1,COP,1194444,1194444,0,0,1115,1193329,0.0009334887194376631
2,ARS,792585,792585,0,0,787,791798,0.0009929534371707766


,merchant_category,records,valid_labels,null_labels,invalid_labels,fraud_records,nonfraud_records,fraud_rate
0,,3396215,3396215,0,0,3287,3392928,0.0009678421419138659
1,Food,256846,256846,0,0,254,256592,0.000988919430320114
2,Services,205124,205124,0,0,196,204928,0.0009555195881515571
3,Other,155029,155029,0,0,155,154874,0.0009998129382244613
4,Transport,154931,154931,0,0,150,154781,0.0009681729285940193
5,Entertainment,153960,153960,0,0,163,153797,0.0010587165497531826
6,Health,102903,102903,0,0,111,102792,0.001078685752601965


## 2. fraud_score vs is_fraud

Cuantiles exactos sobre scores numéricos finitos; NULL/invalid fuera del denominador
numérico, pero contados explícitamente. Buckets fijados por el pedido, sin optimización.
Comparar asociación no equivale a evaluar un modelo ni autorizar un threshold.


In [3]:
fraud_scores(con)
show("fraud_score_by_label_state.csv")
show("fraud_score_buckets.csv")
for dimension in ("transaction_status", "channel", "transaction_type"):
    show(f"fraud_score_by_{dimension}.csv")


2. Score por label, dimensión y buckets fijos completado


,label_state,records,valid_score_count,null_score_count,null_pct,invalid_score_count,valid_labels,fraud_records,mean,median,p75,p90,p95,p99,minimum,maximum
0,False,4420692,3536426,884266,20.002886425926075,0,4420692,0,14.997817706350737,15.0,22.5,27.0,28.5,29.7,0.0,30.0
1,True,4316,3425,891,20.644114921223355,0,4316,4316,49.46448467153282,48.93,74.91,89.94,95.22599999999998,99.18759999999999,0.01,99.99


,score_bucket,transaction_count,valid_labels,fraud_count,nonfraud_count,fraud_rate
0,"01 [0,10)",1179452,1179452,337,1179115,0.00028572591339028633
1,"02 [10,20)",1179243,1179243,359,1178884,0.00030443258938149303
2,"03 [20,30)",1178174,1178174,356,1177818,0.0003021624989178169
3,"04 [30,50)",1312,1312,703,609,0.5358231707317073
4,"05 [50,75)",817,817,817,0,1.0
5,"06 [75,100]",853,853,853,0,1.0
6,07 NULL,885157,885157,891,884266,0.0010066010888463855
7,08 INVALID_NONFINITE,0,0,0,0,
8,09 BELOW_0,0,0,0,0,
9,10 ABOVE_100,0,0,0,0,


,transaction_status,records,valid_score_count,null_score_count,null_pct,invalid_score_count,valid_labels,fraud_records,mean,median,p75,p90,p95,p99,minimum,maximum
0,Approved,4070681,3256373,814308,20.004220424051898,0,4070681,3986,15.031355621730151,15.01,22.51,27.02,28.52,29.72,0.0,99.99
1,Declined,221234,177042,44192,19.975229847130187,0,221234,215,15.040307384688495,15.02,22.52,27.03,28.53,29.72,0.0,99.84
2,Pending,88343,70675,17668,19.99932082904135,0,88343,79,15.007772762645939,15.02,22.48,27.0,28.5,29.72,0.0,97.49
3,Reversed,44750,35761,8989,20.087150837988826,0,44750,36,15.014884930510847,15.02,22.48,26.96,28.51,29.700000000000003,0.0,98.77


,channel,records,valid_score_count,null_score_count,null_pct,invalid_score_count,valid_labels,fraud_records,mean,median,p75,p90,p95,p99,minimum,maximum
0,ATM,1328334,1062528,265806,20.01047929210575,0,1328334,1331,15.03716143009868,15.02,22.52,27.03,28.53,29.72,0.0,99.99
1,App,663414,530878,132536,19.9778720376718,0,663414,634,15.043507152302148,15.04,22.53,27.03,28.53,29.72,0.0,99.93
2,Branch,132495,105991,26504,20.003773727310463,0,132495,143,15.01691294543887,15.03,22.47,26.97,28.53,29.73,0.0,99.84
3,POS,1548161,1238430,309731,20.006381765204008,0,1548161,1467,15.024649233303656,15.0,22.51,27.02,28.52,29.72,0.0,99.97
4,Transfer,89159,71408,17751,19.90937538554717,0,89159,85,15.00404478489801,14.98,22.42,27.0,28.51,29.72,0.0,98.97
5,Web,663445,530616,132829,20.021101975295615,0,663445,656,15.028520606238661,15.02,22.52,27.02,28.52,29.72,0.0,99.84


,transaction_type,records,valid_score_count,null_score_count,null_pct,invalid_score_count,valid_labels,fraud_records,mean,median,p75,p90,p95,p99,minimum,maximum
0,Adjustment,132118,105946,26172,19.80956417747771,0,132118,144,15.041669340984962,15.0,22.55,27.03,28.52,29.72,0.0,99.83
1,Deposit,609409,487082,122327,20.07305438547839,0,609409,607,15.03283892650505,15.01,22.52,27.03,28.53,29.730000000000004,0.0,99.9
2,Payment,738964,590862,148102,20.041842363092112,0,738964,730,15.023827391167227,15.02,22.51,27.02,28.52,29.72,0.0,99.97
3,Purchase,1083406,866642,216764,20.00764256428338,0,1083406,1086,15.033776069011381,15.01,22.51,27.02,28.52,29.72,0.0,99.93
4,Transfer,896438,717843,178595,19.9227386612348,0,896438,798,15.043061769774233,15.03,22.52,27.03,28.52,29.71,0.0,99.97
5,Withdrawal,964673,771476,193197,20.027200927153554,0,964673,951,15.02028752676681,14.99,22.5,27.02,28.53,29.71,0.0,99.99


## 3. Complaint taxonomy

Tablas con denominador total de complaints y cobertura de monto dentro de cada combinación.
Se anotan únicamente categorías observadas, sin definir un cohort ni inferir subtipos ausentes.


In [4]:
taxonomy(con)
show("complaint_case_category.csv")
show("complaint_category_subcategory.csv")
show("complaint_taxonomy.csv", 50)
show("requested_themes_taxonomy_evidence.csv")


3. Taxonomía real y anotaciones exploratorias completadas


,case_type,category,records,pct_of_complaints,claimed_amount_present_count,claimed_amount_present_pct
0,Claim,Branch,3355,5.000372606006409,1261,37.58569299552906
1,Claim,Fees,3357,5.003353454057679,1293,38.51653261840929
2,Claim,Service,3239,4.827483419032715,1207,37.264587835751776
3,Claim,Technical,3312,4.936284372904091,1282,38.707729468599034
4,Claim,Transactions,3335,4.970564125493703,1308,39.220389805097454
5,Complaint,Branch,7969,11.877189060287652,2962,37.169029991215965
6,Complaint,Fees,8171,12.17825471346598,3164,38.72231061069637
7,Complaint,Service,7953,11.853342275877488,3012,37.872500943040365
8,Complaint,Technical,8120,12.10224308815858,3070,37.80788177339902
9,Complaint,Transactions,8239,12.279603547209181,3192,38.74256584536958


,category,subcategory,records,pct_of_complaints,claimed_amount_present_count,claimed_amount_present_pct
0,Branch,Atención en sucursal,11892,17.72412251285491,3735,31.407669021190717
1,Branch,,1469,2.1894328936582457,488,33.21987746766508
2,Fees,Cobro indebido,12194,18.174230568596766,4035,33.09004428407413
3,Fees,,1359,2.0254862508383633,422,31.05224429727741
4,Service,Calidad de servicio,11886,17.715179968701097,3816,32.10499747602221
5,Service,,1308,1.9494746255309636,403,30.810397553516818
6,Technical,Problema con app,12128,18.075862582904836,3923,32.346635883905016
7,Technical,,1279,1.9062523287875401,429,33.54182955433933
8,Transactions,Cargo no reconocido,12297,18.327744243237202,4090,33.26014475075222
9,Transactions,,1283,1.9122140248900812,410,31.95635229929852


,case_type,category,subcategory,records,pct_of_complaints,claimed_amount_present_count,claimed_amount_present_pct
0,Claim,Branch,Atención en sucursal,2989,4.454877412623891,1111,37.16962194713951
1,Claim,Branch,,366,0.5454951933825173,150,40.98360655737705
2,Claim,Fees,Cobro indebido,3026,4.510023101572397,1172,38.730998017184405
3,Claim,Fees,,331,0.49333035248528206,121,36.5558912386707
4,Claim,Service,Calidad de servicio,2906,4.331172218496162,1104,37.99036476256022
5,Claim,Service,,333,0.49631120053655264,103,30.93093093093093
6,Claim,Technical,Problema con app,3006,4.480214621059692,1168,38.85562208915503
7,Claim,Technical,,306,0.45606975184439974,114,37.254901960784316
8,Claim,Transactions,Cargo no reconocido,3014,4.492138013264774,1187,39.3828798938288
9,Claim,Transactions,,321,0.4784261122289291,121,37.69470404984423


,theme,observed_evidence,limitation
0,transaction dispute,Transactions,Broad category; not a final cohort
1,unauthorized charge,Transactions / Cargo no reconocido,Customer taxonomy; not confirmed fraud
2,incorrect charge,Fees / Cobro indebido,May refer to fees rather than transaction amount
3,payment dispute,No dedicated observed category/subcategory,Cannot infer from generic Transactions
4,transfer dispute,No dedicated observed category/subcategory,Cannot infer from generic Transactions
5,ATM/withdrawal dispute,No dedicated observed category/subcategory,Cannot infer from generic Transactions


## 4. claimed_amount usability

Cobertura sobre todas las filas de cada grupo; distribuciones monetarias separadas por
currency. p99/p99.9 describen colas, no justifican borrar extremos. Transactions y Fees
son una vista exploratoria de categorías reales, no un cohort aprobado ni matching.


In [5]:
claimed_amount(con)
show("complaint_claimed_amount_coverage.csv", 60)
show("claimed_amount_checks_overall.csv")
show("claimed_amount_by_currency.csv")
show("claimed_amount_upper_tail_by_currency.csv")
show("potential_transaction_taxonomy_amount.csv", 50)


4. Claimed amount: cobertura condicional y distribuciones por moneda completadas


,scope,dimension,group_value,field,records,present_count,valid_count,null_count,blank_count,invalid_nonnull_count,coverage_pct,present_pct,validity_rule
0,all,overall,ALL,claimed_amount,67095,21751,21751,45344,0,0,32.418212981593264,32.418212981593264,"(try_cast(""claimed_amount"" AS DOUBLE) IS NOT N..."
1,all,overall,ALL,currency,67095,21776,21776,45319,0,0,32.45547358223415,32.45547358223415,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"
2,all,case_type,Claim,claimed_amount,16598,6351,6351,10247,0,0,38.263646222436435,38.263646222436435,"(try_cast(""claimed_amount"" AS DOUBLE) IS NOT N..."
3,all,case_type,Claim,currency,16598,6351,6351,10247,0,0,38.263646222436435,38.263646222436435,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"
4,all,case_type,Complaint,claimed_amount,40452,15400,15400,25052,0,0,38.06981113418372,38.06981113418372,"(try_cast(""claimed_amount"" AS DOUBLE) IS NOT N..."
5,all,case_type,Complaint,currency,40452,15425,15425,25027,0,0,38.13161277563532,38.13161277563532,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"
6,all,case_type,Request,claimed_amount,6761,0,0,6761,0,0,0.0,0.0,"(try_cast(""claimed_amount"" AS DOUBLE) IS NOT N..."
7,all,case_type,Request,currency,6761,0,0,6761,0,0,0.0,0.0,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"
8,all,case_type,Suggestion,claimed_amount,3284,0,0,3284,0,0,0.0,0.0,"(try_cast(""claimed_amount"" AS DOUBLE) IS NOT N..."
9,all,case_type,Suggestion,currency,3284,0,0,3284,0,0,0.0,0.0,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"


,group_value,records,null_amount,invalid_amount,zero_amount,negative_amount,amount_without_currency,currency_without_amount
0,ALL,67095,45344,0,0,0,1040,1065


,currency,records,valid_amounts,mean,median,p75,p90,p95,p99,minimum,maximum
0,ARS,5402,5132,2529.4708067030406,2521.17,3745.425,4500.366000000002,4735.2565,4931.3587,52.31,4997.59
1,COP,5456,5192,2505.3268990755014,2467.195,3753.3475000000003,4513.941,4769.1630000000005,4959.6854,50.27,4999.93
2,MXN,5487,5215,2536.1001073825532,2551.21,3753.63,4510.838,4764.84,4941.4144,50.31,4999.9
3,USD,5431,5172,2566.227376256768,2563.865,3818.745,4549.457,4780.365,4957.4446,50.61,4999.46


,currency,valid_amounts,p99,p999,above_p99,above_p999,maximum
0,ARS,5132,4931.3587,4985.97035,52,6,4997.59
1,COP,5192,4959.6854,4995.68574,52,6,4999.93
2,MXN,5215,4941.4144,4996.839620000001,53,6,4999.9
3,USD,5172,4957.4446,4995.29023,52,6,4999.46


,case_type,category,subcategory,complaints_count,claimed_amount_available,claimed_amount_coverage_pct,currency_coverage_pct,amount_and_currency_available,amount_and_currency_pct
0,Claim,Fees,Cobro indebido,3026,1172,38.730998017184405,38.99537343027099,1114,36.81427627230668
1,Claim,Fees,,331,121,36.5558912386707,35.34743202416919,113,34.13897280966767
2,Claim,Transactions,Cargo no reconocido,3014,1187,39.3828798938288,39.18380889183809,1133,37.59124087591241
3,Claim,Transactions,,321,121,37.69470404984423,37.07165109034268,114,35.51401869158879
4,Complaint,Fees,Cobro indebido,7352,2863,38.9417845484222,39.050598476605,2710,36.86071817192601
5,Complaint,Fees,,819,301,36.75213675213675,36.507936507936506,284,34.676434676434674
6,Complaint,Transactions,Cargo no reconocido,7466,2903,38.88293597642647,38.775783552102865,2774,37.15510313420841
7,Complaint,Transactions,,773,289,37.386804657179816,37.77490297542044,281,36.35187580853816
8,Request,Fees,Cobro indebido,1230,0,0.0,0.0,0,0.0
9,Request,Fees,,137,0,0.0,0.0,0,0.0


## 5. Transaction evidence coverage

valid_count significa presente y compatible con una prueba básica explícita de tipo/rango;
no significa verdad de negocio. NULL y no-nulos inválidos se reportan por separado.
Las 21 columnas se evalúan también por canal, tipo y estado. Unicidad se valida sin joins.


In [6]:
transaction_evidence(con)
show("transaction_identifier_check.csv")
show("transaction_evidence_coverage.csv", 21)


5. Evidencia recuperable y unicidad de transaction_id verificadas


,records,nonnull_ids,distinct_ids
0,4425008,4425008,4425008


,scope,dimension,group_value,field,records,present_count,valid_count,null_count,blank_count,invalid_nonnull_count,coverage_pct,present_pct,validity_rule
0,all,overall,ALL,transaction_id,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""transaction_id"" IS NOT NULL AND trim(""transa..."
1,all,overall,ALL,transaction_date,4425008,4425008,4425008,0,0,0,100.0,100.0,"try_cast(""transaction_date"" AS TIMESTAMP) IS N..."
2,all,overall,ALL,product_id,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""product_id"" IS NOT NULL AND trim(""product_id..."
3,all,overall,ALL,customer_id,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""customer_id"" IS NOT NULL AND trim(""customer_..."
4,all,overall,ALL,transaction_type,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""transaction_type"" IS NOT NULL AND trim(""tran..."
5,all,overall,ALL,transaction_category,4425008,1731488,1731488,2693520,0,0,39.129601573601676,39.129601573601676,"(""transaction_category"" IS NOT NULL AND trim(""..."
6,all,overall,ALL,amount,4425008,4425008,4425008,0,0,0,100.0,100.0,"(try_cast(""amount"" AS DOUBLE) IS NOT NULL AND ..."
7,all,overall,ALL,currency,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""currency"" IS NOT NULL AND trim(""currency"")<>'')"
8,all,overall,ALL,amount_usd,4425008,1887552,1887552,2537456,0,0,42.656465253848125,42.656465253848125,"(try_cast(""amount_usd"" AS DOUBLE) IS NOT NULL ..."
9,all,overall,ALL,channel,4425008,4425008,4425008,0,0,0,100.0,100.0,"(""channel"" IS NOT NULL AND trim(""channel"")<>'')"


Shown 21/357 aggregate rows; full table: transaction_evidence_coverage.csv


## 6. Transaction geography viability

Cobertura de país, ciudad, coordenadas individuales y pares. Rangos físicos no validan
correspondencia con una ubicación real. No geocoding, corrección ni features de anomalía.


In [7]:
geography(con)
show("transaction_geography_coverage.csv", 60)


6. Geografía: cobertura y rangos físicos completados, sin geocoding


,group_value,records,country_count,city_count,latitude_present,longitude_present,lat_lon_pair_present,latitude_numeric,longitude_numeric,latitude_valid,...,country_count_pct,city_count_pct,latitude_present_pct,longitude_present_pct,lat_lon_pair_present_pct,latitude_valid_pct,longitude_valid_pct,lat_lon_pair_valid_pct,latitude_invalid_nonnull,longitude_invalid_nonnull
0,ALL,4425008,4425008,3982397,857328,857293,814415,857328,857293,857328,...,100.0,89.99750960902217,19.374609040254843,19.373818081232848,18.404825482801385,19.374609040254843,19.373818081232848,18.404825482801385,0,0
1,ATM,1328334,1328334,1195368,378312,378284,359211,378312,378284,378312,...,100.0,89.99001757088202,28.480186459128504,28.478078555544013,27.04221980315192,28.480186459128504,28.478078555544013,27.04221980315192,0,0
2,App,663414,663414,596982,0,0,0,0,0,0,...,100.0,89.98634336929881,0.0,0.0,0.0,0.0,0.0,0.0,0,0
3,Branch,132495,132495,119323,37708,37698,35819,37708,37698,37708,...,100.0,90.0584927733122,28.45994188459942,28.45239442997849,27.034227706705913,28.45994188459942,28.45239442997849,27.034227706705913,0,0
4,POS,1548161,1548161,1393254,441308,441311,419385,441308,441311,441308,...,100.0,89.99412851764126,28.505304034916264,28.50549781321193,27.08923684293817,28.505304034916264,28.50549781321193,27.08923684293817,0,0
5,Transfer,89159,89159,80215,0,0,0,0,0,0,...,100.0,89.96848327145885,0.0,0.0,0.0,0.0,0.0,0.0,0,0
6,Web,663445,663445,597255,0,0,0,0,0,0,...,100.0,90.02328753702267,0.0,0.0,0.0,0.0,0.0,0.0,0,0
7,Adjustment,132118,132118,118901,25565,25551,24306,25565,25551,25565,...,100.0,89.99606412449477,19.35012640215565,19.339529814256952,18.397190390408575,19.35012640215565,19.339529814256952,18.397190390408575,0,0
8,Deposit,609409,609409,548405,117799,117760,111793,117799,117760,117799,...,100.0,89.98964570592165,19.33003943164607,19.323639788713326,18.34449442000364,19.33003943164607,19.323639788713326,18.34449442000364,0,0
9,Payment,738964,738964,665012,143386,143285,136222,143386,143285,143386,...,100.0,89.99247595282044,19.4036515987247,19.38998381517909,18.434186239113135,19.4036515987247,19.38998381517909,18.434186239113135,0,0


## 7. transaction_status evidence

Se reutilizan las tablas de fraude/score por estado de secciones 1–2.
Los códigos se conservan como strings: exclusividad observada no demuestra significado.
Los dos porcentajes condicionales muestran denominadores explícitos.


In [8]:
status_evidence(con)
show("status_by_transaction_type.csv", 50)
show("status_by_channel.csv", 50)
show("status_by_response_code.csv")
show("response_code_associations.csv")


7. Estados y códigos como evidencia observable, sin atribuir causas


,transaction_status,transaction_type,records,pct_within_status,pct_within_dimension
0,Approved,Purchase,996168,24.471777572352146,91.94780165515051
1,Approved,Withdrawal,887765,21.808758780164794,92.02755752467417
2,Approved,Transfer,824676,20.258919822015038,91.99476148936122
3,Approved,Payment,679954,16.703691593617872,92.01449597003372
4,Approved,Deposit,560724,13.774697648870054,92.01111240562578
5,Approved,Adjustment,121394,2.982154582980096,91.88301366959838
6,Declined,Purchase,54565,24.663930498928735,5.036431402447467
7,Declined,Withdrawal,47911,21.656255367619806,4.966553433132264
8,Declined,Transfer,44979,20.330961787067086,5.017524915275792
9,Declined,Payment,36851,16.657023784770875,4.986846449894718


,transaction_status,channel,records,pct_within_status,pct_within_dimension
0,Approved,POS,1423963,34.98095281845961,91.97770774486632
1,Approved,ATM,1222086,30.021659766510812,92.00140928411078
2,Approved,App,610492,14.997294064555783,92.02277913942123
3,Approved,Web,610244,14.991201717845245,91.98109865927093
4,Approved,Branch,121978,2.996501076846847,92.06234197516888
5,Approved,Transfer,81918,2.0123905557816983,91.8785540438991
6,Declined,POS,77430,34.99914118083116,5.001417811196639
7,Declined,ATM,66304,29.970076932117124,4.991515688072427
8,Declined,Web,33288,15.046511838144227,5.017446811717625
9,Declined,App,32937,14.887856296952547,4.96477312809196


,transaction_status,response_code,records,pct_within_status,pct_within_dimension
0,Approved,00,3867312,95.00405460413135,100.0
1,Approved,,203369,4.995945395868652,92.00843312989463
2,Declined,51,52788,23.860708571015305,62.70922676677081
3,Declined,14,52711,23.825903794172685,62.40055876503457
4,Declined,54,52527,23.74273393782149,62.62832206602997
5,Declined,05,52246,23.615719102850374,62.09339085582534
6,Declined,,10962,4.954934594140141,4.95944044554433
7,Pending,05,21191,23.987186307913472,25.18510595310253
8,Pending,14,21096,23.879650906127253,24.973955867032863
9,Pending,51,20911,23.670239860543564,24.84111239145155


,response_code,code_records,observed_status_count,dominant_status,dominant_status_records,dominant_status_pct,exclusive_in_this_extract
0,00,3867312,1,Approved,3867312,100.0,True
1,,221033,4,Approved,203369,92.00843312989463,False
2,14,84472,3,Declined,52711,62.40055876503457,False
3,51,84179,3,Declined,52788,62.70922676677081,False
4,05,84141,3,Declined,52246,62.09339085582534,False
5,54,83871,3,Declined,52527,62.62832206602997,False


## 8. Complaint outcome observability

NULL condicionado al lifecycle; resolución no implica cliente correcto ni compensación
implica fraude. La cobertura se exporta por estado/tipo/categoría tanto global como en
la vista exploratoria Transactions/Fees. No se muestran textos crudos de clientes.


In [9]:
outcomes(con, context)
frame = pd.read_csv(ART / "complaint_outcome_coverage.csv", dtype=str, keep_default_na=False)
display(frame[(frame.dimension == "overall") | ((frame.dimension == "status") &
              frame.field.isin(["resolution", "resolution_date", "closing_date"]))])
show("complaint_resolution_days_by_status.csv")
show("complaint_outcome_cutoff_audit.csv")


8. Outcomes observables y límites temporales documentados


,scope,dimension,group_value,field,records,present_count,valid_count,null_count,blank_count,invalid_nonnull_count,coverage_pct,present_pct,validity_rule
0,all,overall,ALL,status,67095,67095,67095,0,0,0,100.0,100.0,"(""status"" IS NOT NULL AND trim(""status"")<>'')"
1,all,overall,ALL,resolution,67095,15310,15310,51785,0,0,22.81839183247634,22.81839183247634,"(""resolution"" IS NOT NULL AND trim(""resolution..."
2,all,overall,ALL,resolution_date,67095,15349,15349,51746,0,0,22.876518369476116,22.876518369476116,"try_cast(""resolution_date"" AS TIMESTAMP) IS NO..."
3,all,overall,ALL,closing_date,67095,2481,2481,64614,0,0,3.6977420076011627,3.6977420076011627,"try_cast(""closing_date"" AS TIMESTAMP) IS NOT NULL"
4,all,overall,ALL,compensation_granted,67095,4641,4641,62454,0,0,6.917057902973396,6.917057902973396,"(try_cast(""compensation_granted"" AS DOUBLE) IS..."
5,all,overall,ALL,resolution_satisfaction,67095,2484,2484,64611,0,0,3.7022132796780682,3.7022132796780682,"(try_cast(""resolution_satisfaction"" AS DOUBLE)..."
6,all,overall,ALL,sla_breached,67095,67095,67095,0,0,0,100.0,100.0,"(CASE lower(trim(""sla_breached"")) WHEN 'true' ..."
7,all,overall,ALL,resolution_days,67095,15363,15363,51732,0,0,22.89738430583501,22.89738430583501,"(try_cast(""resolution_days"" AS DOUBLE) IS NOT ..."
8,all,overall,ALL,priority,67095,67095,67095,0,0,0,100.0,100.0,"(""priority"" IS NOT NULL AND trim(""priority"")<>'')"
10,all,status,Closed,resolution,2609,2477,2477,132,0,0,94.94059026446915,94.94059026446915,"(""resolution"" IS NOT NULL AND trim(""resolution..."


,status,records,numeric_count,mean,median,p75,p90,p95,p99,minimum,maximum
0,Closed,2609,2492,15.578250401284109,15.0,23.0,28.0,29.0,30.0,1.0,30.0
1,Escalated,3321,0,,,,,,,,
2,In Process,26823,0,,,,,,,,
3,Open,20125,0,,,,,,,,
4,Rejected,705,0,,,,,,,,
5,Resolved,13512,12871,15.605935824722243,16.0,23.0,28.0,29.0,30.0,1.0,30.0


,field,status,category,records,null_date,date_at_or_before_cutoff,date_after_cutoff,invalid_date
0,resolution_date,Closed,Branch,551,25,520,6,0
1,resolution_date,Closed,Fees,486,33,444,9,0
2,resolution_date,Closed,Service,493,17,466,10,0
3,resolution_date,Closed,Technical,528,34,489,5,0
4,resolution_date,Closed,Transactions,551,31,514,6,0
5,resolution_date,Escalated,Branch,599,599,,,0
6,resolution_date,Escalated,Fees,700,700,,,0
7,resolution_date,Escalated,Service,692,692,,,0
8,resolution_date,Escalated,Technical,653,653,,,0
9,resolution_date,Escalated,Transactions,677,677,,,0


Shown 30/60 aggregate rows; full table: complaint_outcome_cutoff_audit.csv


## 9. Potential labels — NOT YET APPROVED FOR MODELING

Distribución sobre todas las filas, incluidos NULLs. Bins de compensación describen el signo,
no crean labels de fraude. La recomendación posterior depende de semántica, timing y linkage,
no solamente completitud.


In [10]:
label_distributions(con)
show("potential_label_coverage.csv")
show("potential_label_distributions.csv", 60)


9. Candidatos de labels caracterizados; ninguno aprobado para modelado


,label,records,valid_count,null_count,invalid_nonnull_count,coverage_pct
0,transactions.is_fraud,4425008,4425008,0,0,100.0
1,complaints.status,67095,67095,0,0,100.0
2,complaints.compensation_granted,67095,4641,62454,0,6.917057902973396
3,complaints.resolution_satisfaction,67095,2484,64611,0,3.7022132796780682
4,call_center_interactions.was_resolved,686296,686296,0,0,100.0
5,call_center_interactions.was_escalated,686296,686296,0,0,100.0


,label,observed_class,records,pct_all_rows,interpretation
0,transactions.is_fraud,False,4420692,99.90246345317343,Recorded values; compensation bins are descrip...
1,transactions.is_fraud,True,4316,0.09753654682658201,Recorded values; compensation bins are descrip...
2,complaints.status,Closed,2609,3.88851628288248,Recorded values; compensation bins are descrip...
3,complaints.status,Escalated,3321,4.949698189134809,Recorded values; compensation bins are descrip...
4,complaints.status,In Process,26823,39.97764363961547,Recorded values; compensation bins are descrip...
5,complaints.status,Open,20125,29.994783515910278,Recorded values; compensation bins are descrip...
6,complaints.status,Rejected,705,1.0507489380728818,Recorded values; compensation bins are descrip...
7,complaints.status,Resolved,13512,20.138609434384083,Recorded values; compensation bins are descrip...
8,complaints.compensation_granted,positive,4641,6.917057902973396,Recorded values; compensation bins are descrip...
9,complaints.compensation_granted,,62454,93.0829420970266,Recorded values; compensation bins are descrip...


## 10. Decision matrix & reproducible report

Separación de hechos observados, métricas derivadas, hipótesis e interpretaciones.
Las relaciones ya validadas se reutilizan de los resultados del EDA, sin repetirlo.
HIGH/MEDIUM/LOW/UNUSABLE califican disponibilidad y confiabilidad para el uso descrito.


In [11]:
report = build_report(context)
validate_artifacts()
show("potential_labels_assessment.csv")
show("transaction_dispute_evidence_matrix.csv")
resources.close()
print((ART / "integrity.json").read_text())


10. Matriz y reporte generados: GO WITH CONSTRAINTS para discovery, no adjudicación


Validaciones de artefactos: 769 correctas


,label,records,valid_count,null_count,invalid_nonnull_count,coverage_pct,semantic_meaning,known_leakage_risks,known_linkage_limitations,potential_use,recommendation,class_distribution_artifact
0,transactions.is_fraud,4425008,4425008,0,0,100.0,Recorded boolean fraud flag; confirmation proc...,fraud_score may encode the label or shared syn...,"Unique transaction; ownership validated, but n...",Candidate transaction-level fraud label only a...,requires_validation,potential_label_distributions.csv
1,complaints.status,67095,67095,0,0,100.0,Recorded case lifecycle state; Resolved does n...,Snapshot can reflect future resolution relativ...,No transaction_id; affected product ownership ...,Possible case lifecycle endpoint after definin...,requires_validation,potential_label_distributions.csv
2,complaints.compensation_granted,67095,4641,62454,0,6.917057902973396,Recorded numeric compensation; zero/positive i...,Post-resolution action and policy selection; m...,No verified complaint-transaction linkage; cur...,Observable monetary outcome by currency; unsui...,unsuitable,potential_label_distributions.csv
3,complaints.resolution_satisfaction,67095,2484,64611,0,3.7022132796780682,Recorded resolution satisfaction; scale anchor...,Observed after resolution; response/selection ...,Complaint-level only; no safe transaction linkage,Possible respondent satisfaction endpoint afte...,requires_validation,potential_label_distributions.csv
4,call_center_interactions.was_resolved,686296,686296,0,0,100.0,"Recorded interaction resolution flag, not tran...",Post-contact flag; availability time and resol...,"Customer linkage exists, but no explicit trans...",Possible interaction-level service outcome aft...,requires_validation,potential_label_distributions.csv
5,call_center_interactions.was_escalated,686296,686296,0,0,100.0,Recorded escalation flag; not a normative grou...,Historical human/policy selection; post-contac...,Interaction-level only; cannot transfer to arb...,Possible historical escalation endpoint after ...,requires_validation,potential_label_distributions.csv


,evidence_source,availability,reliability,quantitative_basis,potential_agent_use,main_limitation
0,transaction details,HIGH,HIGH,4425008 unique IDs; date/owner/amount/currency...,"Deterministic lookup, owner consistency and li...","Structural reliability within extract, not aut..."
1,is_fraud,HIGH,LOW,4425008 valid; 4316 positive,Visible recorded flag; conditional future labe...,Undocumented provenance/availability; not conf...
2,fraud_score,MEDIUM,LOW,79.996% finite coverage,Supporting numeric evidence only,Label association and provenance/timing uncert...
3,merchant,LOW,MEDIUM,name 23.259%; category 23.250%,Display observed merchant evidence when present,Incomplete; no independent verification or mer...
4,geography,MEDIUM,LOW,country 100.000%; city 89.998%; valid coordina...,"Optional raw location context, not required MV...",Physical ranges do not validate real location ...
5,complaint taxonomy,HIGH,MEDIUM,67095 cases; actual categories Transactions an...,Exploratory intake language and scope discovery,Coarse recorded categories; no final cohort; n...
6,claimed amount,LOW,MEDIUM,32.418% overall coverage,Possible future candidate-matching evidence wh...,Not a verified transaction amount; no matching...
7,complaint outcome,LOW,LOW,resolution 22.818%; date 22.877%; compensation...,Descriptive case outcomes; conditional future ...,Not dispute-validity ground truth; lifecycle/s...
8,call interactions,HIGH,MEDIUM,686296 interactions; resolution/escalation cov...,Customer-level service evidence; interaction e...,UNUSABLE as direct transaction linkage without...
9,transcripts,LOW,MEDIUM,171321/686296 interactions (24.963%),Future NLP candidate with privacy controls and...,"Partial, selected coverage; transcript-interac..."


Verificación final de fuentes utilizadas y artefactos del EDA general


{
  "used_source_files_sha256_unchanged": 3291,
  "all_raw_files_size_mtime_unchanged": 7671,
  "general_eda_reference_artifacts_sha256_unchanged": true,
  "completed_at": "2026-10-01T23:44:53.297934-05:00"
}



## Findings & implications for Transaction Dispute Discovery

El reporte siguiente responde las seis preguntas requeridas y es también el entregable Markdown.
La decisión permite únicamente discutir la etapa posterior: no la implementa ni la inicia.


In [12]:
display(Markdown(report))

# Transaction Investigation & Dispute — EDA Addendum

## Scope, provenance and denominators

Executed: 2026-10-01T23:44:03.623637-05:00. Primary population: **full observed extract**, not a training cohort.
Inherited feature cutoff: **2026-06-17 inclusive**, from the completed general EDA.
This addendum does not rebuild customer features. Labels and current case status have no documented
availability timestamp: event date before cutoff does **not** make them point-in-time safe.
See [cutoff audit](../artifacts/eda_transaction_dispute/population_cutoff_audit.csv) and
[outcome cutoff audit](../artifacts/eda_transaction_dispute/complaint_outcome_cutoff_audit.csv).
The execution date is not the observation horizon. General EDA observed event dates 2023-06-17 through
2026-06-18, surveys through June 19; process partitions end June 17. Later outcome dates are not
new observation of customer activity. last_updated is not used as a cutoff.

Read-only reuse of existing DuckDB projections; only complaint_id/resolution is additionally read
from raw because the general EDA cache excluded free text. Fallback ingests only the three relevant
domains if the cache is absent. No sampling, no giant pandas frames, no data correction or matching.
The [context](../artifacts/eda_transaction_dispute/analysis_context.json) records hashes and source mode;
[integrity](../artifacts/eda_transaction_dispute/integrity.json) records post-execution checks.

Coverage denominator = all rows of the indicated table/group, including nulls.
Fraud rate = recorded True / valid boolean labels; CSV fraud_rate is a **fraction**, *_pct a percentage.
Numeric quantiles exclude null/unparseable/nonfinite values, whose counts remain explicit.
CSV empty cells mean NULL (not zero); response codes remain strings, including leading zeros.
Basic valid_count tests type/range/nonblank presence, not business truth. No imputation.

## 1. Observed fraud label

**4316 / 4425008 transactions (0.097537%)** carry True.
Valid labels: 4425008; null: 0;
invalid: 0; False: 4420692.
The rare positive class requires future held-out design and provenance review, not model training now.

| dimension | lowest_group | lowest_rate_pct | lowest_records | highest_group | highest_rate_pct | highest_records |
|---|---|---|---|---|---|---|
| transaction_status | Reversed | 0.08045 | 44750 | Approved | 0.09792 | 4070681 |
| transaction_type | Transfer | 0.08902 | 896438 | Adjustment | 0.10899 | 132118 |
| transaction_category | Transport | 0.09335 | 260316 | Health | 0.10683 | 173173 |
| channel | POS | 0.09476 | 1548161 | Branch | 0.10793 | 132495 |
| transaction_country | Colombia | 0.0929 | 1289503 | Mexico | 0.12341 | 40515 |
| currency | COP | 0.09335 | 1194444 | ARS | 0.0993 | 792585 |
| merchant_category | Services | 0.09555 | 205124 | Health | 0.10787 | 102903 |

These min/max rates summarize the requested seven segmentations; exact class counts and group
denominators are in fraud_by_*.csv. Differences are associations, not causal effects; small groups
may have unstable rates. No significance or predictive performance is claimed.

## 2. fraud_score versus recorded label

| label_state | records | valid_score_count | null_pct | mean | median | p75 | p90 | p95 | p99 | minimum | maximum |
|---|---|---|---|---|---|---|---|---|---|---|---|
| False | 4420692 | 3536426 | 20.002886425926075 | 14.997817706350737 | 15.0 | 22.5 | 27.0 | 28.5 | 29.7 | 0.0 | 30.0 |
| True | 4316 | 3425 | 20.644114921223355 | 49.46448467153282 | 48.93 | 74.91 | 89.94 | 95.22599999999998 | 99.18759999999999 | 0.01 | 99.99 |

| score_bucket | transaction_count | valid_labels | fraud_count | nonfraud_count | fraud_rate |
|---|---|---|---|---|---|
| 01 [0,10) | 1179452 | 1179452 | 337 | 1179115 | 0.00028572591339028633 |
| 02 [10,20) | 1179243 | 1179243 | 359 | 1178884 | 0.00030443258938149303 |
| 03 [20,30) | 1178174 | 1178174 | 356 | 1177818 | 0.0003021624989178169 |
| 04 [30,50) | 1312 | 1312 | 703 | 609 | 0.5358231707317073 |
| 05 [50,75) | 817 | 817 | 817 | 0 | 1.0 |
| 06 [75,100] | 853 | 853 | 853 | 0 | 1.0 |
| 07 NULL | 885157 | 885157 | 891 | 884266 | 0.0010066010888463855 |
| 08 INVALID_NONFINITE | 0 | 0 | 0 | 0 | NULL |
| 09 BELOW_0 | 0 | 0 | 0 | 0 | NULL |
| 10 ABOVE_100 | 0 | 0 | 0 | 0 | NULL |

Observed score ranges overlap: True minimum 0.01, False maximum 30.0.
The positive group also extends to 99.99. This separation in the upper tail is
**a signal and a possible label-construction/leakage concern**, not proof of predictive utility.
No calibrated probabilities, threshold optimization or score-based fraud rule is approved.
Score-by-status/channel/type exports retain counts and missingness.

## 3. Complaint taxonomy — observed, not a final cohort

| category | subcategory | records | pct_of_complaints | claimed_amount_present_count | claimed_amount_present_pct |
|---|---|---|---|---|---|
| Fees | Cobro indebido | 12194 | 18.174230568596766 | 4035 | 33.09004428407413 |
| Fees | NULL | 1359 | 2.0254862508383633 | 422 | 31.05224429727741 |
| Transactions | Cargo no reconocido | 12297 | 18.327744243237202 | 4090 | 33.26014475075222 |
| Transactions | NULL | 1283 | 1.9122140248900812 | 410 | 31.95635229929852 |

Transactions / Cargo no reconocido and Fees / Cobro indebido are descriptively relevant.
Null subcategories and all case types remain included in the tables.
There are no dedicated observed payment-, transfer- or ATM-dispute categories; generic Transactions
cannot establish those subtypes. Full two- and three-way tables include the other real categories:
Branch, Service and Technical. Annotations are hypotheses about relevance, not a new taxonomy.

## 4. Claimed amount usability

Overall finite amount coverage: **32.418%**;
currency coverage: **32.455%**.
Joint finite amount + known currency: **20711/67095
(30.868%)**. All observed non-null amounts are finite;
the joint calculation removes only the explicitly counted unknown-currency cases from this metric.

| group_value | records | valid_count | null_count | coverage_pct |
|---|---|---|---|---|
| Claim | 16598 | 6351 | 10247 | 38.263646222436435 |
| Complaint | 40452 | 15400 | 25052 | 38.06981113418372 |
| Request | 6761 | 0 | 6761 | 0.0 |
| Suggestion | 3284 | 0 | 3284 | 0.0 |

| group_value | records | null_amount | invalid_amount | zero_amount | negative_amount | amount_without_currency | currency_without_amount |
|---|---|---|---|---|---|---|---|
| ALL | 67095 | 45344 | 0 | 0 | 0 | 1040 | 1065 |

| currency | records | valid_amounts | mean | median | p75 | p90 | p95 | p99 | minimum | maximum |
|---|---|---|---|---|---|---|---|---|---|---|
| ARS | 5402 | 5132 | 2529.4708067030406 | 2521.17 | 3745.425 | 4500.366000000002 | 4735.2565 | 4931.3587 | 52.31 | 4997.59 |
| COP | 5456 | 5192 | 2505.3268990755014 | 2467.195 | 3753.3475000000003 | 4513.941 | 4769.1630000000005 | 4959.6854 | 50.27 | 4999.93 |
| MXN | 5487 | 5215 | 2536.1001073825532 | 2551.21 | 3753.63 | 4510.838 | 4764.84 | 4941.4144 | 50.31 | 4999.9 |
| USD | 5431 | 5172 | 2566.227376256768 | 2563.865 | 3818.745 | 4549.457 | 4780.365 | 4957.4446 | 50.61 | 4999.46 |

No currencies are pooled or converted. Amounts with unknown currency remain in coverage/checks
but are excluded from monetary quantiles: their units cannot safely be combined.
Upper-tail counts above each currency's p99/p99.9 are
descriptive, not invalid-value tests, and all extremes are retained. See claimed_amount_upper_tail_by_currency.csv.
Case-type conditional absence should not be mislabeled as universal data corruption.
The potential_transaction_taxonomy_amount.csv export gives case/category/subcategory-specific amount
and currency coverage. A recorded amount could narrow future candidates, but cannot identify a
transaction. Joint amount-and-currency coverage is separately exported for each combination.
A recorded amount cannot identify a
transaction or justify matching by itself.

## 5. Transaction evidence retrievability

| field | records | valid_count | null_count | invalid_nonnull_count | coverage_pct |
|---|---|---|---|---|---|
| transaction_id | 4425008 | 4425008 | 0 | 0 | 100.0 |
| transaction_date | 4425008 | 4425008 | 0 | 0 | 100.0 |
| product_id | 4425008 | 4425008 | 0 | 0 | 100.0 |
| customer_id | 4425008 | 4425008 | 0 | 0 | 100.0 |
| transaction_type | 4425008 | 4425008 | 0 | 0 | 100.0 |
| transaction_category | 4425008 | 1731488 | 2693520 | 0 | 39.129601573601676 |
| amount | 4425008 | 4425008 | 0 | 0 | 100.0 |
| currency | 4425008 | 4425008 | 0 | 0 | 100.0 |
| amount_usd | 4425008 | 1887552 | 2537456 | 0 | 42.656465253848125 |
| channel | 4425008 | 4425008 | 0 | 0 | 100.0 |
| branch_id | 4425008 | 1387932 | 3037076 | 0 | 31.365638209015668 |
| merchant_name | 4425008 | 1029234 | 3395774 | 0 | 23.259483372685427 |
| merchant_category | 4425008 | 1028793 | 3396215 | 0 | 23.249517289008292 |
| transaction_country | 4425008 | 4425008 | 0 | 0 | 100.0 |
| transaction_city | 4425008 | 3982397 | 442611 | 0 | 89.99750960902217 |
| transaction_status | 4425008 | 4425008 | 0 | 0 | 100.0 |
| response_code | 4425008 | 4203975 | 221033 | 0 | 95.00491298546805 |
| is_fraud | 4425008 | 4425008 | 0 | 0 | 100.0 |
| fraud_score | 4425008 | 3539851 | 885157 | 0 | 79.99648814194234 |
| latitude | 4425008 | 857328 | 3567680 | 0 | 19.374609040254843 |
| longitude | 4425008 | 857293 | 3567715 | 0 | 19.373818081232848 |

Uniqueness is checked in transaction_identifier_check.csv. Coverage is also exported by channel,
transaction_type and transaction_status. Amount is interpreted only with its currency; amount_usd
is not silently filled from amount, including USD rows. Ownership consistency is inherited from
validated transactions → products → customers, not from coincident identifiers.

## 6. Geography viability

| records | country_count_pct | city_count_pct | latitude_present_pct | longitude_present_pct | lat_lon_pair_present_pct | lat_lon_pair_valid_pct | latitude_out_of_range | longitude_out_of_range | latitude_invalid_nonnull | longitude_invalid_nonnull |
|---|---|---|---|---|---|---|---|---|---|---|
| 4425008 | 100.0 | 89.99750960902217 | 19.374609040254843 | 19.373818081232848 | 18.404825482801385 | 18.404825482801385 | 0 | 0 | 0 | 0 |

Per-channel/type/country coverage is exported. Valid physical ranges do not establish that coordinates
agree with the recorded city or country. **Exclude geographic anomaly logic from the MVP**; show
optional raw geography only with missingness and provenance. No geocoding or location correction.

## 7. Status and response-code evidence

| response_code | code_records | observed_status_count | dominant_status | dominant_status_records | dominant_status_pct | exclusive_in_this_extract |
|---|---|---|---|---|---|---|
| 00 | 3867312 | 1 | Approved | 3867312 | 100.0 | True |
| NULL | 221033 | 4 | Approved | 203369 | 92.00843312989463 | False |
| 14 | 84472 | 3 | Declined | 52711 | 62.40055876503457 | False |
| 51 | 84179 | 3 | Declined | 52788 | 62.70922676677081 | False |
| 05 | 84141 | 3 | Declined | 52246 | 62.09339085582534 | False |
| 54 | 83871 | 3 | Declined | 52527 | 62.62832206602997 | False |

Observed code exclusivity is limited to this extract. Codes are not translated into undocumented
decline causes. Status-by-code/type/channel tables provide both P(dimension|status) and
P(status|dimension); missing codes are a separate group. Status-by-fraud and score-by-status
reuse sections 1–2. Approved does not itself prove settlement, and Reversed does not establish
that a complaint was upheld.

## 8. Complaint outcome observability

| field | records | valid_count | null_count | coverage_pct |
|---|---|---|---|---|
| status | 67095 | 67095 | 0 | 100.0 |
| resolution | 67095 | 15310 | 51785 | 22.81839183247634 |
| resolution_date | 67095 | 15349 | 51746 | 22.876518369476116 |
| closing_date | 67095 | 2481 | 64614 | 3.6977420076011627 |
| compensation_granted | 67095 | 4641 | 62454 | 6.917057902973396 |
| resolution_satisfaction | 67095 | 2484 | 64611 | 3.7022132796780682 |
| sla_breached | 67095 | 67095 | 0 | 100.0 |
| resolution_days | 67095 | 15363 | 51732 | 22.89738430583501 |
| priority | 67095 | 67095 | 0 | 100.0 |

complaint_outcome_coverage.csv conditions coverage on status/case_type/category/subcategory for all complaints
and separately the exploratory Transactions-or-Fees subset (not a finalized cohort).
Resolution/closing nulls may reflect lifecycle, not a missing-data error.
resolution_satisfaction has respondent/closure selection; compensation is a post-case monetary
observation, **not fraud truth**. Resolved does not mean the customer was right.
Resolution text is measured for presence without exposing raw customer text.
Outcome dates after cutoff are reported, not included as available historical features.
No existing field establishes a verified dispute verdict linked to a specific transaction.

## 9. Potential labels — NOT YET APPROVED FOR MODELING

| label | coverage_pct | semantic_meaning | known_leakage_risks | known_linkage_limitations | potential_use | recommendation |
|---|---|---|---|---|---|---|
| transactions.is_fraud | 100.0 | Recorded boolean fraud flag; confirmation process, source and availability time undocumented | fraud_score may encode the label or shared synthetic generation; post-event flag; severe imbalance | Unique transaction; ownership validated, but no complaint-transaction link | Candidate transaction-level fraud label only after provenance, timing and held-out validation | requires_validation |
| complaints.status | 100.0 | Recorded case lifecycle state; Resolved does not mean customer was right | Snapshot can reflect future resolution relative to creation and cutoff | No transaction_id; affected product ownership is inconsistent; origin interaction absent | Possible case lifecycle endpoint after defining time horizon and censoring | requires_validation |
| complaints.compensation_granted | 6.917057902973396 | Recorded numeric compensation; zero/positive is not a fraud verdict | Post-resolution action and policy selection; missingness depends on lifecycle | No verified complaint-transaction linkage; currencies cannot be pooled | Observable monetary outcome by currency; unsuitable as ground truth of fraud or dispute validity | unsuitable |
| complaints.resolution_satisfaction | 3.7022132796780682 | Recorded resolution satisfaction; scale anchors and collection timing undocumented | Observed after resolution; response/selection bias; cannot stand for all complaints | Complaint-level only; no safe transaction linkage | Possible respondent satisfaction endpoint after scale and sampling validation | requires_validation |
| call_center_interactions.was_resolved | 100.0 | Recorded interaction resolution flag, not transaction dispute adjudication | Post-contact flag; availability time and resolution criteria unknown | Customer linkage exists, but no explicit transaction linkage or complaint-origin link | Possible interaction-level service outcome after semantic validation | requires_validation |
| call_center_interactions.was_escalated | 100.0 | Recorded escalation flag; not a normative ground-truth decision to escalate | Historical human/policy selection; post-contact flag and timing unknown | Interaction-level only; cannot transfer to arbitrary customer transactions | Possible historical escalation endpoint after policy and timestamp validation | requires_validation |

Class distributions, including NULL and descriptive compensation sign bins, are in
[potential_label_distributions.csv](../artifacts/eda_transaction_dispute/potential_label_distributions.csv).
Compensation bins are not newly approved labels. None of these candidates is approved for training.
The recommendation unsuitable refers specifically to compensation as fraud/dispute-validity ground truth.
Future held-out evaluation requires provenance, outcome timing, censoring, a task definition and
customer/time-aware split design; no models or split optimization are performed here.

## 10. Decision matrix

Availability ratings: HIGH = broadly recoverable core evidence; MEDIUM = usable partial fields;
LOW = sparse/selected for the intended use; UNUSABLE = absent or invalid required linkage.
Reliability rates the narrow use described, not source truth certification. HIGH structural transaction
reliability does not authorize payments or legal conclusions; LOW label reliability reflects absent
provenance, not evidence that recorded values are false.

| evidence_source | availability | reliability | quantitative_basis | potential_agent_use | main_limitation |
|---|---|---|---|---|---|
| transaction details | HIGH | HIGH | 4425008 unique IDs; date/owner/amount/currency coverage 100.000%/100.000%/100.000%/100.000% | Deterministic lookup, owner consistency and literal amount/status evidence | Structural reliability within extract, not authorization or settlement proof; amount_usd partial |
| is_fraud | HIGH | LOW | 4425008 valid; 4316 positive | Visible recorded flag; conditional future label candidate | Undocumented provenance/availability; not confirmed fraud or decision authority |
| fraud_score | MEDIUM | LOW | 79.996% finite coverage | Supporting numeric evidence only | Label association and provenance/timing uncertainty; no operational threshold justified |
| merchant | LOW | MEDIUM | name 23.259%; category 23.250% | Display observed merchant evidence when present | Incomplete; no independent verification or merchant identity reference |
| geography | MEDIUM | LOW | country 100.000%; city 89.998%; valid coordinate pair 18.405% | Optional raw location context, not required MVP decision logic | Physical ranges do not validate real location or country-coordinate coherence |
| complaint taxonomy | HIGH | MEDIUM | 67095 cases; actual categories Transactions and Fees | Exploratory intake language and scope discovery | Coarse recorded categories; no final cohort; no transfer/ATM-specific taxonomy |
| claimed amount | LOW | MEDIUM | 32.418% overall coverage | Possible future candidate-matching evidence when amount AND currency exist | Not a verified transaction amount; no matching performed; missingness by case type |
| complaint outcome | LOW | LOW | resolution 22.818%; date 22.877%; compensation 6.917%; satisfaction 3.702% | Descriptive case outcomes; conditional future offline evaluation | Not dispute-validity ground truth; lifecycle/selection/timing and linkage limitations |
| call interactions | HIGH | MEDIUM | 686296 interactions; resolution/escalation coverage in labels CSV | Customer-level service evidence; interaction endpoints requiring semantic validation | UNUSABLE as direct transaction linkage without additional evidence |
| transcripts | LOW | MEDIUM | 171321/686296 interactions (24.963%) | Future NLP candidate with privacy controls and evidence attribution | Partial, selected coverage; transcript-interaction consistency is not a transaction link |

## Preserved general EDA findings

| validation | total | eligible | linked | orphan | inconsistent | consistent | consistent_pct |
|---|---|---|---|---|---|---|---|
| transaction_product_customer | 4425008 | 4425008 | 4425008 | 0 | 0 | 4425008 | 100.0 |
| transcript_customer | 171321 | 171321 | 171321 | 0 | 0 | 171321 | 100.0 |
| transcript_agent | 171321 | 171321 | 171321 | 0 | 0 | 171321 | 100.0 |
| survey_customer | 212759 | 212759 | 212759 | 0 | 0 | 212759 | 100.0 |
| survey_agent | 212759 | 212759 | 212759 | 0 | 0 | 212759 | 100.0 |
| complaint_product_customer | 67095 | 44570 | 44570 | 0 | 44570 | 0 | 0.0 |

All 44,570 non-null complaint affected-product references point to existing products **owned by
different customers**. No repair or owner reassignment is allowed. complaint.origin_interaction_id
is entirely null. Customer and agent branch joins remain unreliable.
Digital anonymous share: 23.977%; do not attach those events to customers.
Transcript coverage: 24.963% of all interactions and
24.960% of calls. Verified transcript → interaction identity does
not imply a transcript → transaction or complaint link.
Source: [unchanged General EDA findings](eda/EDA_FINDINGS.md).

## Hypotheses and pending validation — not findings

1. fraud_score and is_fraud may share label generation or post-event information; inspect provenance.
2. Transactions / Cargo no reconocido and Fees / Cobro indebido may support different dispute-intake
   patterns; no final cohort or transaction matching is defined.
3. Claimed amount plus currency may help candidate retrieval where present, but temporal, ownership
   and linkage constraints must be independently established.
4. Complaint outcomes and call flags might support separate service-quality endpoints, not fraud
   correctness. Any future evaluation must separate service completion from adjudication.

## Required conclusions

### 1. Does the dataset support a Transaction Investigation workflow?

**Yes, with constraints:** 4425008 individually identified transactions,
validated transaction/product/customer ownership, literal amounts/currencies/dates/statuses and
partially available merchant/score evidence support scoped lookup and evidence collection.
This establishes data feasibility, not a production system, independent truth verification or an
automatic financial decision.

### 2. Does the dataset support a Transaction Dispute workflow?

- **Dispute intake:** conditionally supported by observed case taxonomy and amount/currency fields.
  Missing fields must remain unknown; no complete intake policy is documented.
- **Transaction investigation:** supported as evidence retrieval and structural validation.
- **Dispute matching:** **not validated**. No direct complaint-transaction key and broken affected-product
  ownership; future discovery must investigate feasibility, not assume a match.
- **Dispute decision/resolution:** **not supported for autonomous adjudication or gold-label evaluation**.
  No verified linked verdict, policy basis, label provenance or point-in-time outcome availability.

### 3. Which signals are reliable enough for deterministic logic?

Within this immutable extract: unique transaction_id lookup, transaction/customer/product ownership
consistency, observed date, amount paired with currency, and literal transaction_status.
Response codes and is_fraud can be returned **as recorded evidence only**, not interpreted as causes,
confirmed fraud or permission to reimburse. Missing evidence and inconsistent ownership require
explicit unknown/error states; no silent repair. Dataset ownership is not authentication/authorization.

### 4. Which signals are appropriate as AI/ML inputs?

- **Raw evidence:** transaction fields with coverage/provenance, optional merchant/location,
  customer-level service context only where linkage is explicit.
- **Derived feature candidates:** existing cutoff-aware independent aggregates, missingness indicators,
  documented summaries; do not add post-event labels/snapshot outcomes as contemporaneous predictors.
- **Label candidates:** is_fraud and service/case outcomes require validation listed above;
  compensation is not fraud ground truth. No approved label or model is delivered.
- **LLM/NLP candidates:** limited transcripts tied to interactions and complaint text might support
  future evidence extraction with privacy/grounding controls. Neither NLP nor LLM analysis was run,
  and these texts are not linked to transactions by this addendum.

### 5. Which relationships remain unsafe to assume?

Complaint → affected product ownership; complaint → origin interaction; complaint → transaction;
anonymous digital event → customer; same-customer call/transcript → specific transaction;
customer/agent branch links; score/flag → verified fraud; compensation or Resolved → justified dispute;
response code → undocumented decline cause; event date → label availability.

### 6. Can we proceed to transaction_dispute_discovery?

**GO WITH CONSTRAINTS.** Proceed only to scoped discovery of transaction evidence retrieval and
dispute-intake feasibility, preserving known broken links and unknown semantics. Before matching,
automation or ML, require verified linkage, label provenance/timing, policy documentation and
an auditable evaluation target. Autonomous dispute decisions remain NO-GO on present evidence.
No stage-03 notebook, journeys, matching, demo customer selection, models or agent architecture
were implemented in this task.
